# Modeling

This notebook trains the models for our research question: which recorded crash conditions are linked to injuries, and how well can models predict them? The code is in `src/data3800_gruppe_7/modeling.py`, and the notebook runs it one step at a time with the reasoning behind each choice.

The fitted models and their predictions on the test set are saved in `output/models/`, where the evaluation notebook reads them. Running `uv run python -m data3800_gruppe_7.modeling` from the project root does the same without the notebook.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from data3800_gruppe_7.modeling import (
    build_preprocessor,
    load_model_data,
    predict_test_set,
    save_models,
    save_predictions,
    split_train_test,
    train_models,
)

## Data

We use the cleaned crashes from 2025 (see `cleaning.py`). One row is one crash reported to the police, so the data says nothing about trips that ended without a crash. The target `injury` is 1 if at least one person was injured.

The models get five columns as input. The scale of each column decides how we can use it:

| Column | Scale | Used as |
| --- | --- | --- |
| `WEATHER_CONDITION` | nominal | category |
| `LIGHTING_CONDITION` | nominal | category |
| `ROADWAY_SURFACE_COND` | nominal | category |
| `POSTED_SPEED_LIMIT` | ratio | number |
| `CRASH_HOUR` | interval, discrete | category |

The injury counts, `MOST_SEVERE_INJURY`, `CRASH_TYPE` and `DAMAGE` are never used as input. They are recorded after the crash and would give away the answer.

In [ ]:
X, y, crash_ids = load_model_data()
print(f"{len(X)} crashes, {y.mean():.1%} with at least one injury")
X.head()

## Training set and test set

We keep 20% of the crashes aside as a test set. The models never see these crashes during training, so the test results show how well they do on new crashes. `stratify=y` gives both sets the same share of injuries, and `random_state=42` makes the split identical every time the code runs. The evaluation notebook can call `split_train_test` to get the same test set.

In [ ]:
X_train, X_test, y_train, y_test = split_train_test(X, y)
pd.DataFrame(
    {
        "crashes": [len(y_train), len(y_test)],
        "injury share": [y_train.mean(), y_test.mean()],
    },
    index=["training set", "test set"],
)

## Preprocessing

The models only work with numbers. Weather, lighting and road surface are nominal, so the only information in them is which category a crash belongs to. We one-hot encode them: every category gets its own column of 0s and 1s. The hour is on an interval scale, but as a number it would tell the logistic regression that the risk rises or falls steadily from midnight to 23:00. We have no reason to believe that, so the hour is one-hot encoded too. The speed limit is on a ratio scale and stays a number.

After cleaning, 198 crashes have no speed limit, because a limit of 0 was set to missing. We fill these in with the median of the training set, which is 30 mph. We use the median because it is robust to outliers, and the column has a few, such as limits of 1 to 3 mph. Then the column is scaled to mean 0 and standard deviation 1. Scaling matters for logistic regression and changes nothing for the trees, so all four models can share the same preprocessing.

Each model gets its own pipeline with the preprocessing as the first step. The median and the list of categories are then learned from the training set only, so nothing from the test set leaks into training.

In [ ]:
build_preprocessor()

## Models

| Name | Model and settings | Why we use it |
| --- | --- | --- |
| `baseline` | `DummyClassifier(strategy="most_frequent")` | Always predicts "no injury". The other models have to beat it. |
| `logistic_regression` | `LogisticRegression()` | A linear model whose coefficients are easy to interpret. |
| `decision_tree` | `DecisionTreeClassifier(max_depth=8)` | Can pick up combinations of conditions. The depth limit keeps it from memorizing the training set. |
| `random_forest` | `RandomForestClassifier(n_estimators=200, min_samples_leaf=50)` | Averages 200 trees, which is usually more stable than a single tree. Every leaf needs at least 50 crashes, so no rule rests on a handful of cases. |

The last three models also get `class_weight="balanced"`. Only 16.4% of the crashes have an injury, so a model can reach 83.6% accuracy by always predicting "no injury", which is what the baseline does, and it would still miss every injury. With balanced weights, a mistake on an injury crash counts about five times as much during training as a mistake on a crash without injuries, which gives the models a reason to find them. The price is more false alarms.

In [ ]:
pipelines = train_models(X_train, y_train)

## A first check

The full evaluation is in the evaluation notebook. Here we only check that training worked as expected, by looking at how large a share of the test set each model predicts as injury.

In [ ]:
predictions = predict_test_set(pipelines, X_test, y_test, crash_ids)
predictions.groupby("model", sort=False)["y_pred"].mean().rename(
    "predicted injury share"
)

The baseline never predicts an injury. Because of the balanced class weights, the other three predict injury for 60 to 68% of the test crashes, far more than the 16.4% that actually had one.

By Bayes' theorem, the chance that a flagged crash really had an injury, P(injury | predicted injury), depends on the base rate of 16.4% as well as on how many of the injuries the model finds. A model can therefore find most of the injuries and still be wrong about most of the crashes it flags. The evaluation notebook measures both: recall is the share of the real injuries a model finds, and precision is P(injury | predicted injury).

## Saving the results

Each fitted pipeline is saved with joblib as `output/models/<name>.joblib`, and the predictions go to `output/models/predictions.csv` with one row per crash and model:

| Column | Content |
| --- | --- |
| `crash_record_id` | ID of the crash |
| `model` | Name of the model |
| `y_true` | 1 if the crash had an injury, otherwise 0 |
| `y_pred` | The model's prediction, 0 or 1 |
| `y_proba` | The predicted probability of injury |

The files are large and take a few seconds to rebuild, so `output/models/` is in `.gitignore`. Run this notebook or the module before the evaluation notebook.

In [ ]:
save_models(pipelines)
save_predictions(predictions)
predictions.head()

## Limitations

- The split is random. For the final report we plan to train on January to August, tune on September and October and test on November and December, which is closer to how a model would be used.
- The tree settings are reasonable starting values that we have not tuned yet.
- A condition that goes together with injuries in this data does not necessarily cause them.